<a href="https://colab.research.google.com/github/dylanhogg/jupyter-experiments/blob/fine-tuning/notebooks/finetuning/llama-factory/examples/Finetune_Llama3_with_LLaMA_Factory_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Llama-3 with LLaMA Factory v5 (llm-address)

Updated: TODO

<a href="https://colab.research.google.com/github/dylanhogg/jupyter-experiments/blob/fine-tuning/notebooks/finetuning/llama-factory/examples/Finetune_Llama3_with_LLaMA_Factory_v5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Github source: https://github.com/dylanhogg/jupyter-experiments/blob/fine-tuning/notebooks/finetuning/llama-factory/examples/Finetune_Llama3_with_LLaMA_Factory_v5.ipynb

LLaMA Factory Project homepage: https://github.com/hiyouga/LLaMA-Factory

Training data: https://huggingface.co/datasets/dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct

v0.2 models:
TODO

## Imports

In [ ]:
import os
import hashlib
import json
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import files, userdata

## HF Auth

HF auth can be required for gated models you need to be granted access to.

In [ ]:
try:
  hf_token = userdata.get("HF_TOKEN")
  print("Loading HF_TOKEN from Colab Secrets...")
  os.environ["HF_TOKEN"] = hf_token
except Exception as e:
  print("No HF_TOKEN Colab Secret set in environment, fall back to interactive login...")
  !hf auth login

## Variables

In [ ]:
# https://huggingface.co/unsloth/models?sort=downloads&search=llama
# model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
# model_name_or_path="unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
# model_name_or_path="unsloth/Llama-3.2-1B-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model

# https://huggingface.co/meta-llama/models?sort=downloads&search=llama
# model_name_or_path="meta-llama/Llama-3.2-3B-Instruct"
model_name_or_path="meta-llama/Llama-3.2-1B-Instruct"

## Install Dependencies

In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]  # TODO: swap out for uv

### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory")

## Fine-tune model via Command Line

It takes ~30min for training.

In [ ]:
# Local backup of original dataset_info.json
!cp data/dataset_info.json data/dataset_info_original.json

In [ ]:
# Create new custom dataset_info.json
# Ref: https://github.com/hiyouga/LLaMA-Factory/blob/main/data/dataset_info.json
dataset_info = {
    "gnaf-2022-structured-training-1000000-v0-instruct-train": {
        "hf_hub_url": "dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct",
        "split": "train",
        "columns": {
          "prompt": "instruction",
          "query": "input",
          "response": "output"
        },
    },
    "gnaf-2022-structured-training-1000000-v0-instruct-test": {
        "hf_hub_url": "dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct",
        "split": "test",
        "columns": {
          "prompt": "instruction",
          "query": "input",
          "response": "output"
        },
    }
}
json.dump(dataset_info, open("data/dataset_info.json", "w", encoding="utf-8"), indent=2)

In [ ]:
!cat data/dataset_info.json

In [ ]:
# Train

# Examples: https://github.com/hiyouga/LLaMA-Factory/tree/main/examples/train_lora

num_train_epochs=50.0
dataset = "gnaf-2022-structured-training-1000000-v0-instruct-train"  # see custom dataset_info.json rendered above
eval_dataset = "gnaf-2022-structured-training-1000000-v0-instruct-test"  # see custom dataset_info.json rendered above

template = "llama3"  # use llama3 prompt template
finetuning_type = "lora"  # use LoRA adapters to save memory

train_args = dict(
  stage="sft",                                               # do supervised fine-tuning
  do_train=True,
  model_name_or_path=model_name_or_path,
  dataset=dataset,
  eval_dataset=eval_dataset,
  template=template,
  finetuning_type=finetuning_type,
  lora_target="all",                                         # attach LoRA adapters to all linear layers
  output_dir="llama3_lora",                                  # the path to save LoRA adapters
  plot_loss=True,
  per_device_train_batch_size=2,                             # the micro batch size
  gradient_accumulation_steps=4,                             # the gradient accumulation steps
  lr_scheduler_type="cosine",                                # use cosine learning rate scheduler
  logging_steps=5,                                           # log every 5 steps
  warmup_ratio=0.1,                                          # use warmup scheduler
  save_steps=1000,                                           # save checkpoint every 1000 steps
  learning_rate=5e-5,                                        # the learning rate
  num_train_epochs=num_train_epochs,
  max_samples=500,                                           # use 500 examples in each dataset
  max_grad_norm=1.0,                                         # clip gradient norm to 1.0
  loraplus_lr_ratio=16.0,                                    # use LoRA+ algorithm with lambda=16.0
  fp16=True,                                                 # use float16 mixed precision training
  report_to="none",                                          # disable wandb logging
)

train_args_hash = hashlib.sha256(str(train_args).encode("utf-8")).hexdigest()[0:7]
aest_now = datetime.now(ZoneInfo("Australia/Sydney")).strftime('%Y%m%d-%H%M%S')
model_version = "v0.2"
target_model_name = f"dylanhogg/gnaf-structured-address-{model_version}-{train_args_hash}-{aest_now}"
print(f"{target_model_name=}")
print(f"{train_args=}")
print(f"{train_args_hash=}")

json.dump(train_args, open("train_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/
!llamafactory-cli train train_llama3.json

In [ ]:
!ls /content/LLaMA-Factory/llama3_lora

In [ ]:
!zip -r llama_lora_results.zip /content/LLaMA-Factory/llama3_lora

In [ ]:
!ls -lha /content/LLaMA-Factory/llama_lora_results.zip

In [ ]:
files.download("/content/LLaMA-Factory/llama_lora_results.zip")

## Infer the fine-tuned model

In [ ]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

%cd /content/LLaMA-Factory/

args = dict(
  # model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  model_name_or_path=model_name_or_path,
  adapter_name_or_path="llama3_lora",                        # load the saved LoRA adapters
  template=template,                                         # same to the one in training
  finetuning_type=finetuning_type,                           # same to the one in training
)
chat_model = ChatModel(args)

def build_user_message(instruction: str, query: str = "") -> str:
    """
    Reconstruct the same input format the model saw in training.
    """
    if query:
        return f"{instruction}\n{query}\n"
    return f"{instruction}\n"

messages = []
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
while True:
  query = input("\nUser: ")
  if query.strip() == "exit":
    break
  if query.strip() == "clear":
    messages = []
    torch_gc()
    print("History has been removed.")
    continue

  instruction = "Translate a text address into stuctured json."  # TODO: get from training dataset!
  user_message = build_user_message(instruction, query)
  messages.append({"role": "user", "content": user_message})
  print("Assistant: ", end="", flush=True)

  response = ""
  for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()

## Merge the LoRA adapter and optionally upload model

NOTE: the Colab free version has merely 12GB RAM, where merging LoRA of a 8B model needs at least 18GB RAM, thus you **cannot** perform it in the free version.

In [ ]:
print(f"{target_model_name=}")

In [ ]:
# Create model repo on Huggingface as private, ready to push to after model merge
!hf repo create {target_model_name} --repo-type model --private

In [ ]:
# Merge and export model

args = dict(
  model_name_or_path=model_name_or_path,                    # use official non-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="llama3_lora",                       # load the saved LoRA adapters
  template=template,                                        # same to the one in training
  finetuning_type="lora",                                   # same to the one in training

  # export
  export_dir="llama3_lora_merged",                          # the path to save the merged model
  export_size=2,                                            # the file shard size (in GB) of the merged model
  export_device="cpu",                                      # the device used in export, can be chosen from `cpu` and `auto`
  export_hub_model_id=target_model_name                     # the Hugging Face hub ID to upload model
)

json.dump(args, open("merge_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

!llamafactory-cli export merge_llama3.json

## Review merged model outputs

In [ ]:
!ls -lha /content/LLaMA-Factory/llama3_lora

In [ ]:
!ls -lha /content/LLaMA-Factory/llama3_lora_merged

In [ ]:
# Automatically generated model README.md
# TODO: replace sections in README
!cat /content/LLaMA-Factory/llama3_lora/README.md

In [ ]:
# Upload autogenerated model README
!hf upload {target_model_name} /content/LLaMA-Factory/llama3_lora/README.md /README.md

## Zip merged model for download

In [ ]:
# NOTE: can be slow since it's merged in the large base model
!zip -r llama3_lora_merged.zip /content/LLaMA-Factory/llama3_lora_merged

In [ ]:
!ls -lha /content/LLaMA-Factory/llama3_lora_merged.zip

In [ ]:
files.download("/content/LLaMA-Factory/llama3_lora_merged.zip")

In [ ]:
print("Done!")